# 01 - Bronze Ingestion

## Purpose

Read Retail Banking CSV files from Lakehouse Files, add ingestion metadata, preserve the source structure, and write raw Bronze Delta tables.

## Concepts covered

- Lakehouse Files as the landing area
- CSV ingestion with Spark
- Bronze layer design
- Ingestion timestamps, source file names, and batch IDs
- Row count logging and audit tables
- Repeatable demo loads using overwrite mode

## Prerequisites

- Run 00_setup_lakehouse.ipynb or manually upload sample CSVs to Files/retail_banking/source.
- Attach this notebook to the target Fabric Lakehouse.

## Expected output

Bronze Delta tables for customers, accounts, products, branches, transactions, plus bronze_ingestion_audit.

In [ ]:
from datetime import datetime, timezone
from typing import Dict, List

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

SOURCE_BASE_PATH = "Files/retail_banking/source"
ENTITIES = ["customers", "accounts", "products", "branches", "transactions"]
BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
LOAD_MODE = "overwrite"  # Repeatable learning demo. Production ingestion is often append or merge.
SOURCE_SYSTEM = "retail_banking_csv"

def log_info(message: str) -> None:
    print(f"[INFO] {datetime.now(timezone.utc).isoformat()} | {message}")

def log_error(message: str) -> None:
    print(f"[ERROR] {datetime.now(timezone.utc).isoformat()} | {message}")

log_info(f"Starting Bronze ingestion batch {BATCH_ID}")

## Bronze design rule

Bronze tables intentionally keep source values as strings. Business cleanup, type casting, deduplication, and referential validation happen in Silver. This keeps Bronze useful for replay and troubleshooting.

In [ ]:
def source_file(entity_name: str) -> str:
    return f"{SOURCE_BASE_PATH}/{entity_name}.csv"

def bronze_table(entity_name: str) -> str:
    return f"bronze_{entity_name}"

def read_source_csv(entity_name: str) -> DataFrame:
    path = source_file(entity_name)
    log_info(f"Reading source file: {path}")
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "false")
        .option("escape", '"')
        .load(path)
    )

def add_ingestion_metadata(df: DataFrame, entity_name: str) -> DataFrame:
    return (
        df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file_name", F.input_file_name())
        .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("_source_entity", F.lit(entity_name))
        .withColumn("_batch_id", F.lit(BATCH_ID))
        .withColumn("_bronze_load_date", F.current_date())
    )

def write_bronze_table(df: DataFrame, entity_name: str) -> int:
    table_name = bronze_table(entity_name)
    row_count = df.count()
    df.write.format("delta").mode(LOAD_MODE).option("overwriteSchema", "true").saveAsTable(table_name)
    log_info(f"Wrote {table_name} with {row_count} rows")
    return row_count

## Ingest each entity

The loop records source row counts, target row counts, status, and error messages. In a production Fabric Data Pipeline, this audit output supports monitoring and retry decisions.

In [ ]:
audit_rows: List[Dict[str, object]] = []
failures: List[str] = []

for entity in ENTITIES:
    started_at = datetime.now(timezone.utc).isoformat()
    try:
        raw_df = read_source_csv(entity)
        source_row_count = raw_df.count()
        bronze_df = add_ingestion_metadata(raw_df, entity)
        target_row_count = write_bronze_table(bronze_df, entity)
        status = "SUCCESS" if source_row_count == target_row_count else "ROW_COUNT_MISMATCH"

        audit_rows.append({
            "batch_id": BATCH_ID,
            "entity_name": entity,
            "source_path": source_file(entity),
            "target_table": bronze_table(entity),
            "source_row_count": source_row_count,
            "target_row_count": target_row_count,
            "status": status,
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "error_message": None,
        })

        if status != "SUCCESS":
            failures.append(f"{entity}: source={source_row_count}, target={target_row_count}")

    except Exception as exc:
        message = str(exc)[:1000]
        log_error(f"Failed to ingest {entity}: {message}")
        failures.append(f"{entity}: {message}")
        audit_rows.append({
            "batch_id": BATCH_ID,
            "entity_name": entity,
            "source_path": source_file(entity),
            "target_table": bronze_table(entity),
            "source_row_count": 0,
            "target_row_count": 0,
            "status": "FAILED",
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "error_message": message,
        })

audit_df = spark.createDataFrame(audit_rows)
audit_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("bronze_ingestion_audit")
display(audit_df.orderBy("entity_name"))

if failures:
    raise RuntimeError("Bronze ingestion completed with failures: " + "; ".join(failures))

## Validate Bronze row counts

In [ ]:
validation_rows = []

for entity in ENTITIES:
    source_count = spark.read.format("csv").option("header", "true").option("inferSchema", "false").load(source_file(entity)).count()
    bronze_count = spark.table(bronze_table(entity)).count()
    validation_rows.append({
        "entity_name": entity,
        "source_row_count": source_count,
        "bronze_row_count": bronze_count,
        "status": "PASS" if source_count == bronze_count else "FAIL",
    })

bronze_validation_df = spark.createDataFrame(validation_rows)
display(bronze_validation_df.orderBy("entity_name"))

failed_count = bronze_validation_df.filter(F.col("status") == "FAIL").count()
if failed_count > 0:
    raise RuntimeError(f"Bronze row count validation failed for {failed_count} entities")

## Preview Bronze data

In [ ]:
for entity in ENTITIES:
    table_name = bronze_table(entity)
    log_info(f"Previewing {table_name}")
    display(spark.table(table_name).limit(5))

## Next steps

Run 02_silver_transformation.ipynb to clean data types, standardize values, remove duplicates, and validate business keys.